# Experiment 4: DP utility and membership leakage with disjoint 2023 shadows

Compare a fresh non-private ACS 2023 target (epsilon = infinity) and five private
targets at epsilon = 0.5, 1, 2, 4, 8, with delta = 1e-6 fixed. Use 100,000
target training records, all 48 expanded inputs and nominal PINCP > 50,000 labels.
There is no missing-feature reconstruction, PFV or uncertainty aggregation.

Reserve every target train, validation, test, MIA member and MIA nonmember ID
before constructing the unused ACS 2023 auxiliary pool (approximately 73,000
eligible records; report the actual count). Each of 16 non-private shadows gets
its own stratified 80/20 train/test split. Fit each shadow's imputation, scaling
and encoding strictly on its own training rows. Retain four RF, four gradient
boosting, four XGBoost and four generic MLP shadows. MLP shadows train a fixed
50 epochs without validation selection; the OUT test split is never used to
choose epochs. Generic shadow architectures and settings are not target-matched.

Retain the accuracy-focused Exp 4 target recipe: hidden layers 512/256/128,
dropout 0.1, Adam learning rate 0.0003, weight decay 0.0001, maximum 30 epochs,
public-validation accuracy selection (loss breaks ties), patience 5 and best
weight restoration. Logical batches remain 256, physical batches at most 32.
Private targets use Opacus clipping, Gaussian noise, Poisson sampling and the
strict RDP accounting calibration of the original accuracy-focused study.

Train the complete-feature four-signal RF membership classifier once on balanced
2,000 IN and 2,000 OUT examples per shadow, with grouped auxiliary record-ID
train/validation/test roles. Freeze that classifier and its auxiliary-selected
thresholds across all six targets. Blind walks and commuting-field perturbations
are identical for shadows and targets. All shadows are non-private: this remains
a transfer attack, rather than an attack adapted separately to each DP budget.

Plot target test accuracy, MIA accuracy and low-FPR membership leakage against
equally spaced epsilon categories, with a non-private horizontal reference.
Report both evaluation ROC TPR at FPR <= 1% and TPR/actual FPR at the frozen
auxiliary threshold. The evaluation ROC summary uses target membership labels
only to summarize results; it does not select a deployed attack threshold.

Same-year shadows control the source-year mismatch. They do not prove the attack
is optimal, guarantee monotonic curves, or make near-chance MIA a privacy proof.
One run per condition has no repeated-run error bars. Each target's DP guarantee
uses its fixed maximum-schedule bound; stopped-time epsilon is diagnostic only.

No notebook cells have been run. No models, checkpoints, logs or row-level target
scores are saved. Final metrics, public auxiliary row assignments, validation
histories, protocol and PNG/SVG figures are research outputs, not DP releases.


## Dependencies and fixed study configuration
Use the project virtual environment. If the dependency check below fails, install
`python -m pip install -r requirements.txt` in that environment before running.
Installation is separate from executing this notebook.

Private training uses Opacus per-example gradient clipping and Gaussian noise
with Adam (DP-Adam), Poisson sampling and an RDP accountant. Physical batches
are capped at 32 to limit per-sample-gradient memory; logical batches remain 256.
The existing four paper-inspired attack features, including blind walks, are
retained with consistent numeric perturbations for shadows and targets.


In [1]:
from pathlib import Path
from importlib.util import find_spec
import csv
import gc
import importlib.metadata
import json
import math

missing_dependencies = [name for name in ['opacus', 'matplotlib', 'xgboost'] if find_spec(name) is None]
if missing_dependencies:
    raise ImportError('Missing ' + ', '.join(missing_dependencies)
        + '. Install in this notebook kernel: python -m pip install -r requirements.txt')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from src.data import split_acs_data, subset_target_training, TARGET
import src.expanded_data as expanded_data_module
importlib.reload(expanded_data_module)
from src.expanded_data import EXPANDED_FEATURES, load_expanded_year, reference_split_config
from src.dp_experiment import (
    prepare_public_preprocessing, train_dp_study_target, plot_budget_accuracy,
)
from src.training import evaluate_model
import src.shadow_models as shadow_models_module
importlib.reload(shadow_models_module)
from src.shadow_models import (train_shadow_model, summarize_shadow_models,
    PAPER_SHADOW_ALGORITHMS, MLP_SHADOW_CONFIG)
from src.same_year_mia import disjoint_same_year_pool, split_same_year_shadow
from src.dp_same_year import plot_low_fpr_budget
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features
from src.notebook_output import CellProgress, quiet_output, suppress_sklearn_parallel_warning

suppress_sklearn_parallel_warning()
torch.set_num_threads(4)
SEED = 42
TARGET_YEAR = 2023
AUXILIARY_YEAR = 2023
PUBLIC_PREPROCESSING_YEAR = 2023
TRAIN_SIZE = 100_000
PUBLIC_PREPROCESSING_SIZE = 10_000
EPSILON_BUDGETS = (0.5, 1, 2, 4, 8)
DELTA = 1e-6
assert 0 < DELTA < 1 / TRAIN_SIZE
assert len(EXPANDED_FEATURES) == 48
N_SHADOW_MODELS = 16
SHADOW_RECORDS_PER_CLASS = 2000
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS + ('mlp',)
SHADOW_MLP_CONFIG = dict(MLP_SHADOW_CONFIG,
    early_stopping=False, restore_best=False)
assert N_SHADOW_MODELS == 4 * len(SHADOW_ALGORITHMS)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.1,
    max_epochs=30, patience=5, min_delta=0.0, batch_size=256,
    max_physical_batch_size=32, learning_rate=3e-4, weight_decay=1e-4,
    max_grad_norm=1.0, seed=SEED, device='cpu',
    selection_metric='accuracy', strict_accounting=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
    batch_size=512, seed=SEED)
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
RUN_DIR = Path('artifacts/exp_4_2023_2023_dp_e0p5_1_2_4_8_d1e6_s16_train80_test20_public2023')

for year in sorted({TARGET_YEAR, AUXILIARY_YEAR, PUBLIC_PREPROCESSING_YEAR}):
    filename = 'ss14pca.csv' if year == 2014 else 'psam_p06.csv'
    raw_path = Path('data/raw/folktables') / str(year) / '1-Year' / filename
    with raw_path.open(newline='', encoding='utf-8-sig') as handle:
        columns = set(next(csv.reader(handle)))
    missing_columns = set(EXPANDED_FEATURES + ['PINCP', 'PWGTP']) - columns
    if missing_columns:
        raise ValueError(f'{raw_path}: missing columns {sorted(missing_columns)}')

versions = {name: importlib.metadata.version(name) for name in
    ['torch', 'opacus', 'numpy', 'pandas', 'scikit-learn', 'xgboost', 'matplotlib']}
print('Budgets:', EPSILON_BUDGETS, '| Constant delta:', DELTA)
print('Epoch cap:', TRAIN_CONFIG['max_epochs'], '| Patience:', TRAIN_CONFIG['patience'])
print('Shadow split: 80/20; fixed-schedule MLP shadows, no shadow validation.')
print('Target public preprocessing year:', PUBLIC_PREPROCESSING_YEAR)
print('Complete-feature MIA; four features including blind walks; no reconstruction.')
display(pd.Series(versions, name='package_version'))


Budgets: (0.5, 1, 2, 4, 8) | Constant delta: 1e-06
Epoch cap: 30 | Patience: 5
Shadow split: 80/20; fixed-schedule MLP shadows, no shadow validation.
Target public preprocessing year: 2023
Complete-feature MIA; four features including blind walks; no reconstruction.


torch           2.15.0
opacus           1.6.0
numpy            2.5.4
pandas           3.1.1
scikit-learn     1.9.1
xgboost          3.4.1
matplotlib      3.11.2
Name: package_version, dtype: str

## Fixed public preprocessing and disjoint target/shadow data
Use the original stratified target partitions and 100,000-record training subset.
The fixed target validation set is public and selects epochs only. Test and MIA
records never select model settings or thresholds. Exclude the union of all five
target partitions before constructing the same-year auxiliary pool. Report and
save zero-overlap audits, exact public pool IDs and the actual pool size.

Fit the target preprocessor on a seeded 10,000-row public subset of the
unused 2023 auxiliary pool, then freeze it for all six targets. These public
preprocessing rows remain eligible for shadow train/test splits so all roughly
73,000 unused rows remain in the shadow pool. Overlap with shadow records is
intentional; they are disjoint from every target partition. Each shadow still
fits its own independent preprocessor on its own training split only; target
preprocessing is never reused to fit a shadow. Shadow OUT rows never fit their
own shadow's representation or model.

The record-level DP statement concerns each target's protected training records,
conditional on the fixed research partition and independently available public
data. Public preprocessing and public validation do not consume that target
training budget. Public shadow models themselves have no DP guarantee.


In [2]:
target_source = load_expanded_year(TARGET_YEAR, income_base_year=None)
reference_partitions = split_acs_data(target_source, **SPLIT_CONFIG)
target_partitions = subset_target_training(reference_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(target_partitions[name].index, reference_partitions[name].index)
assert len(target_partitions['train']) == TRAIN_SIZE
assert len(target_partitions['mia_members']) == len(target_partitions['mia_nonmembers'])
shadow_auxiliary, auxiliary_disjointness_audit = disjoint_same_year_pool(
    target_source, target_partitions)
assert list(shadow_auxiliary.columns) == list(target_source.columns)
assert auxiliary_disjointness_audit['overlapping_record_ids'].eq(0).all()
_, public_preprocessing = train_test_split(shadow_auxiliary,
    test_size=PUBLIC_PREPROCESSING_SIZE, stratify=shadow_auxiliary[TARGET], random_state=SEED)
public_preprocessing = public_preprocessing.sort_index().copy()
assert set(public_preprocessing.index).issubset(shadow_auxiliary.index)

target_preprocessor, target_arrays = prepare_public_preprocessing(public_preprocessing, target_partitions)
partition_counts = {name: len(frame) for name, frame in target_partitions.items()}
public_preprocessing_shadow_overlap = len(set(public_preprocessing.index) & set(shadow_auxiliary.index))
display(auxiliary_disjointness_audit)
display(pd.Series({**partition_counts, 'public_preprocessing': len(public_preprocessing),
    'disjoint_2023_shadow_pool': len(shadow_auxiliary)}, name='records'))
print('Public preprocessing / shadow pool overlap:', public_preprocessing_shadow_overlap)
print('Encoded input dimensions shared by all targets:', target_arrays['train']['X'].shape[1])
del target_source, reference_partitions
gc.collect()


,target_partition,target_records,shadow_pool_records,overlapping_record_ids
0,validation,10000,73278,0
1,test,10000,73278,0
2,mia_nonmembers,10000,73278,0
3,train,100000,73278,0
4,mia_members,10000,73278,0


validation                    10000
test                          10000
mia_nonmembers                10000
train                        100000
mia_members                   10000
public_preprocessing          10000
disjoint_2023_shadow_pool     73278
Name: records, dtype: int64

Public preprocessing / shadow pool overlap: 10000
Encoded input dimensions shared by all targets: 2336


129

## Fit and freeze the same-year shadow attack
Train 16 non-private ACS 2023 shadows: four each RF, gradient boosting, XGBoost
and generic MLP. Each shadow independently splits the entire unused pool 80/20
with seed 42 + 10 * shadow_index. There is no shadow validation partition. MLPs
use the existing 128/64 architecture and optimizer, a fixed 50 epochs, and final
weights. Tree settings are unchanged. Each preprocessor fits training rows only.

Sample 2,000 IN training rows and 2,000 OUT test rows per shadow for scoring,
keeping attack examples balanced without querying every shadow training row.
Group repeated record IDs into attack train/validation/test roles; those meta
roles are distinct from a shadow's 80/20 model-data split. The RF attack and
accuracy/low-FPR thresholds are selected using auxiliary data only and frozen
before target evaluation. Release each shadow after extracting its features.

This controls source year but remains a non-private, mixed-model transfer attack.
It does not establish the strongest possible DP-aware attack. Do not interpret
weak auxiliary held-out performance as evidence of target privacy.


In [3]:
def fit_and_query_shadow(auxiliary, name, algorithm, seed):
    partitions = split_same_year_shadow(auxiliary, seed=seed, early_stopping=False)
    assert set(partitions) == {'train', 'nonmembers'}
    with quiet_output():
        result = train_shadow_model(auxiliary, algorithm, seed=seed, schema='expanded_acs',
            parameters=SHADOW_MLP_CONFIG if algorithm == 'mlp' else None, partitions=partitions)
        result['attack_data']['shadow_model'] = name
        features, audits = collect_shadow_paper_features({name: result}, auxiliary,
            per_class=SHADOW_RECORDS_PER_CLASS, config=WALK_CONFIG, seed=SEED,
            include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
    summary = summarize_shadow_models({name: result})
    counts = features['membership'].value_counts()
    assert counts[0] == counts[1] == SHADOW_RECORDS_PER_CLASS
    audit = dict(audits[name], seed=seed,
        train_records=len(partitions['train']), out_test_records=len(partitions['nonmembers']),
        validation_records=0, sampled_in=int(counts[1]), sampled_out=int(counts[0]),
        epochs_run=result['metadata'].get('epochs_run'),
        early_stopping=result['metadata'].get('early_stopping', False))
    return features, audit, summary

shadow_progress = CellProgress()
shadow_feature_parts, shadow_summary_parts, shadow_query_audits, shadow_seeds = [], [], {}, {}
for index in range(N_SHADOW_MODELS):
    algorithm = SHADOW_ALGORITHMS[index % len(SHADOW_ALGORITHMS)]
    name = f'{algorithm}_{index + 1:02d}'
    shadow_seeds[name] = SEED + 10 * index
    shadow_progress.update(f'Shadow {index + 1}/{N_SHADOW_MODELS}: training/querying {name}')
    features, audit, summary = fit_and_query_shadow(shadow_auxiliary, name, algorithm, shadow_seeds[name])
    shadow_feature_parts.append(features)
    shadow_summary_parts.append(summary)
    shadow_query_audits[name] = audit
    del features, audit, summary
    gc.collect()

shadow_attack_data = pd.concat(shadow_feature_parts, ignore_index=True)
shadow_income_metrics = pd.concat(shadow_summary_parts)
assert len(shadow_attack_data) == N_SHADOW_MODELS * 2 * SHADOW_RECORDS_PER_CLASS
assert shadow_attack_data['shadow_model'].nunique() == N_SHADOW_MODELS
with quiet_output():
    attack_result = train_membership_classifier(shadow_attack_data, seed=SEED,
        algorithm='random_forest', feature_set='paper')
split_ids = {name: set(frame['record_id']) for name, frame in attack_result['splits'].items()}
assert split_ids['train'].isdisjoint(split_ids['validation'] | split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])
auxiliary_attack_metrics = pd.DataFrame(attack_result['metrics']).T
shadow_progress.update('All shadows scored; auxiliary attack and thresholds frozen')
display(auxiliary_attack_metrics[['accuracy', 'balanced_accuracy', 'roc_auc', 'tpr_at_1pct_fpr_roc']].round(4))
del shadow_feature_parts, shadow_summary_parts
gc.collect()


,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc
train,0.5949,0.5953,0.6609,0.0496
validation,0.5478,0.5455,0.5690,0.0217
test,0.5292,0.5302,0.5589,0.0177


120

## Train and evaluate one target at a time
The helper below defines the work; the following run cell performs it. Each
model starts from scratch with the same architecture/initialization seed and
regularization. Opacus uses independent, nondeterministically seeded sampling
and noise generators; their seeds are not saved. Highest public-validation
accuracy selects the returned epoch; lower log loss breaks exact ties. The test set, private training metrics and MIA
labels do not influence training or model selection.

Each private run calibrates noise for the **fixed maximum 30-epoch schedule**.
The fixed-schedule epsilon bound covers the early-stopped, validation-selected
model. Epsilon at the actual stopping time is recorded separately as a diagnostic;
it is not advertised as a tighter adaptive-stopping guarantee. Restoring an
earlier epoch does not reset the privacy expenditure. Delta remains 1e-6.

After selection, measure train/test utility and the signed train-minus-test
accuracy gap. Query the same balanced target MIA records using the frozen
four-feature attack. Only one target model and its query table are held at a time.

Before the first update, recalibrate noise using the private loader's exact
logical step count and the same conservative rate used by the Opacus accountant.
Record both the actual sampler rate and the accountant/bound rate. Target architecture, regularization, learning rate, epoch selection,
splits, physical/logical batch sizes, DP accounting and blind walks
follow accuracy-focused Exp 4. This variant changes auxiliary source
year, shadow splits/schedule and the declared budget grid. The public
preprocessing source is recorded explicitly in this notebook.


In [4]:
target_progress = CellProgress()

def run_target_condition(epsilon):
    condition = 'non_private' if epsilon is None else f'epsilon_{epsilon:g}'
    target_progress.update(f'{condition}: training from scratch')
    with quiet_output():
        model, history, metadata = train_dp_study_target(
            target_arrays['train']['X'], target_arrays['train']['y'],
            target_arrays['validation']['X'], target_arrays['validation']['y'],
            epsilon=epsilon, delta=DELTA, progress=target_progress.update, **TRAIN_CONFIG)
    target_progress.update(f'{condition}: evaluating selected epoch {metadata["best_epoch"]}')
    income_metrics = {name: evaluate_model(model, target_arrays[name]['X'], target_arrays[name]['y'])
        for name in ['train', 'validation', 'test']}
    target_progress.update(f'{condition}: querying complete-feature membership attack')
    with quiet_output():
        attack_features, query_audit = collect_target_paper_features(model, target_preprocessor,
            target_partitions, config=WALK_CONFIG, seed=SEED,
            include_blind_walk=True, numeric_protocol=EXPANDED_NUMERIC_WALK)
        mia_metrics, _ = evaluate_attack(attack_result['classifier'], attack_features,
            threshold=attack_result['accuracy_threshold'],
            low_fpr_threshold=attack_result['low_fpr_threshold'])
    assert mia_metrics['records'] == 2 * mia_metrics['members']
    row = dict(condition=condition, private=epsilon is not None,
        epsilon_requested=epsilon, delta=DELTA if epsilon is not None else None,
        epsilon_fixed_schedule_bound=metadata['epsilon_fixed_schedule_bound'],
        epsilon_at_stop_diagnostic=metadata['epsilon_at_stop_diagnostic'],
        noise_multiplier=metadata['noise_multiplier'],
        sampler_sample_rate=metadata['sampler_sample_rate'],
        accountant_sample_rate=metadata['accountant_sample_rate'],
        bound_sample_rate=metadata['bound_sample_rate'],
        selection_metric=metadata['selection_metric'], epochs_run=metadata['epochs_run'],
        best_epoch=metadata['best_epoch'], training_seconds=metadata['training_seconds'],
        target_train_accuracy=income_metrics['train']['accuracy'],
        target_validation_accuracy=income_metrics['validation']['accuracy'],
        target_test_accuracy=income_metrics['test']['accuracy'],
        target_test_roc_auc=income_metrics['test']['roc_auc'],
        target_test_log_loss=income_metrics['test']['log_loss'],
        generalization_gap=income_metrics['train']['accuracy'] - income_metrics['test']['accuracy'],
        mia_accuracy=mia_metrics['accuracy'], mia_balanced_accuracy=mia_metrics['balanced_accuracy'],
        mia_roc_auc=mia_metrics['roc_auc'], mia_tpr_at_1pct_fpr_roc=mia_metrics['tpr_at_1pct_fpr_roc'],
        mia_tpr_at_shadow_threshold=mia_metrics['tpr_at_shadow_threshold'],
        mia_fpr_at_shadow_threshold=mia_metrics['fpr_at_shadow_threshold'])
    metadata.update(income_metrics=income_metrics, membership_metrics=mia_metrics, query_audit=query_audit)
    # Return metrics only: no fitted target, private row scores or model files.
    return row, history, metadata


### Run six matched conditions
This cell trains a fresh non-private control (epsilon = infinity) and five
private targets at epsilon = 0.5, 1, 2, 4, 8. It is an experiment and requires
explicit approval to execute. No historical targets or cached scores are used.
Report every budget without selecting one using target test or MIA outcomes.
The three known DP research warnings are shown once; errors remain visible.


In [ ]:
# Reload presentation support only; keep existing data, shadows and attack.
import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
from src.notebook_output import compact_dp_warnings

study_rows, training_histories, target_metadata = [], {}, {}
with compact_dp_warnings():
    for epsilon in (None,) + EPSILON_BUDGETS:
        row, history, metadata = run_target_condition(epsilon)
        study_rows.append(row)
        training_histories[row['condition']] = history
        target_metadata[row['condition']] = metadata
        gc.collect()
study_results = pd.DataFrame(study_rows)
assert len(study_results) == 1 + len(EPSILON_BUDGETS)
assert study_results.loc[study_results['private'], 'delta'].eq(DELTA).all()
target_progress.update('All six conditions completed; final metrics ready')


C:\Users\syedk\Desktop\Semester 1\Responsible AI - ECE 750\Project\MF-MIA\.venv\Lib\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\syedk\Desktop\Semester 1\Responsible AI - ECE 750\Project\MF-MIA\.venv\Lib\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
C:\Users\syedk\Desktop\Semester 1\Responsible AI - ECE 750\Project\MF-MIA\src\dp_experiment.py:219: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backw

## Compare utility, generalization and membership leakage
Accuracies below are fractions; graphs show percentages. The generalization gap
is training accuracy minus test accuracy. A small gap with poor test accuracy can
indicate underfitting, so both are reported. Ordinary MIA accuracy uses one
auxiliary-validation-selected threshold, frozen across targets. ROC-AUC and
TPR@1%FPR supplement accuracy so threshold transfer is visible.


In [ ]:
display(study_results.set_index('condition')[[
    'epsilon_requested', 'epsilon_fixed_schedule_bound', 'epsilon_at_stop_diagnostic',
    'delta', 'epochs_run', 'best_epoch', 'target_train_accuracy', 'target_test_accuracy',
    'generalization_gap', 'mia_accuracy', 'mia_roc_auc', 'mia_tpr_at_1pct_fpr_roc',
    'mia_tpr_at_shadow_threshold', 'mia_fpr_at_shadow_threshold'
]].round(6))

display(study_results.loc[study_results['private'], [
    'condition', 'sampler_sample_rate', 'accountant_sample_rate', 'bound_sample_rate',
    'noise_multiplier', 'epsilon_fixed_schedule_bound'
]].set_index('condition'))


## Graph 1: target model accuracy versus privacy budget
Use held-out **income-prediction test accuracy**. Epsilon labels are categories
with equal spacing, not numeric coordinates. The dashed line is the matched
non-private control. Budget labels describe requested maximum-schedule budgets;
actual accountant values and stopping epochs are available in the result table.


In [ ]:
figure, axes = plot_budget_accuracy(study_results, EPSILON_BUDGETS,
    metric='target_test_accuracy', output_dir=RUN_DIR, delta=DELTA)
display(figure)
plt.close(figure)


## Graph 2: membership inference accuracy versus privacy budget
Use complete-feature **target MIA accuracy**, with the same categorical epsilon
axis. The non-private control and 50% chance level are horizontal references.
Chance is 50% because members and nonmembers are equally represented. Neither
curve is forced to be monotonic. One run per condition does not support error
bars or a claim that a near-chance attack proves privacy.


In [ ]:
figure, axes = plot_budget_accuracy(study_results, EPSILON_BUDGETS,
    metric='mia_accuracy', output_dir=RUN_DIR, delta=DELTA)
display(figure)
plt.close(figure)


## Graph 3: low-FPR membership leakage versus privacy budget
Left: evaluation ROC TPR at empirical FPR <= 1%, summarized using target labels.
Right: TPR and actual target FPR at the single frozen auxiliary-validation cutoff.
The fixed cutoff is not recalibrated on target records and need not achieve 1%
target FPR. Report the actual FPR even when the TPR appears favorable. Epsilon
uses categorical spacing; the non-private control is a horizontal reference.
These single-run measurements do not force or establish monotonic privacy gains.


In [ ]:
figure, axes = plot_low_fpr_budget(study_results, EPSILON_BUDGETS,
    output_dir=RUN_DIR, delta=DELTA)
display(figure)
plt.close(figure)


## Save final research results and interpretation limits
Save final CSV metrics, selected training histories, protocol and all three figures.
These research outputs include private-data evaluation statistics and are **not
DP-sanitized releases**. No private training rows, membership-labelled prediction
tables, fitted models, checkpoints or logs are written by this notebook.

The stated epsilon/delta bound concerns each individual target inference model
with its fixed public preprocessor. Releasing several models trained on the same
records requires privacy composition; the non-private control has no DP
guarantee. `secure_mode=False` is explicitly a research setting. Early stopping
uses public validation only. The recorded realised-stop accountant value is a
diagnostic; the fixed maximum-schedule bound is the claimed bound.

This study has one initialization seed per condition and independently sampled
DP noise. It uses transferred mixed tree/MLP-shadow attacks, same-year Census shadow data,
a fixed public vocabulary, record-level protection, and a fixed
research partition. Repeated runs and attacks matched to DP targets are follow-up
checks. Lower measured MIA does not by itself demonstrate stronger protection.


In [ ]:
def json_value(value):
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    if isinstance(value, np.generic):
        return json_value(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return str(value)
    return value

RUN_DIR.mkdir(parents=True, exist_ok=True)
study_results.to_csv(RUN_DIR / 'dp_target_and_mia_metrics.csv', index=False)
auxiliary_attack_metrics.to_csv(RUN_DIR / 'auxiliary_attack_metrics.csv')
shadow_income_metrics.to_csv(RUN_DIR / 'shadow_income_metrics.csv')
auxiliary_disjointness_audit.to_csv(RUN_DIR / 'auxiliary_disjointness_audit.csv', index=False)
pd.Series(shadow_auxiliary.index, name='record_id').to_csv(RUN_DIR / 'auxiliary_pool_ids.csv', index=False)
pd.Series(public_preprocessing.index, name='record_id').to_csv(RUN_DIR / 'public_preprocessing_ids.csv', index=False)
for condition, history in training_histories.items():
    pd.DataFrame(history).to_csv(RUN_DIR / f'{condition}_public_validation_history.csv', index=False)
protocol = dict(study_variant='same-year 2023 shadows, 80/20, accuracy-focused DP targets',
    accounting_calibration='Exact private-loader steps; conservative Opacus accountant rate',
    execution_status='complete', target_year=TARGET_YEAR, auxiliary_year=AUXILIARY_YEAR,
    epsilon_budgets=EPSILON_BUDGETS, delta=DELTA, train_size=TRAIN_SIZE,
    feature_count=len(EXPANDED_FEATURES), features=EXPANDED_FEATURES,
    label='nominal PINCP > 50000', split_config=SPLIT_CONFIG, target_partition_sizes=partition_counts,
    public_preprocessing_records=PUBLIC_PREPROCESSING_SIZE,
    preprocessing_scope='Seeded public 2023 subset of unused target-source records; may overlap shadow splits; shared by all targets',
    public_preprocessing_year=PUBLIC_PREPROCESSING_YEAR,
    public_preprocessing_shadow_overlap=public_preprocessing_shadow_overlap,
    auxiliary_pool_records=len(shadow_auxiliary),
    auxiliary_pool_source='Unused ACS 2023 source records excluding every target partition',
    auxiliary_disjointness_audit=auxiliary_disjointness_audit.to_dict('records'),
    shadow_split_fractions=dict(train=0.8, nonmembers=0.2),
    shadow_mlp_selection='Fixed 50 epochs; final weights; no validation/test selection',
    public_validation='Fixed 2023 validation split outside protected training records',
    training_config=TRAIN_CONFIG, target_results=target_metadata,
    privacy_scope='Individual target model; record-level DP conditional on fixed partition/public data',
    privacy_bound='Fixed maximum-schedule epsilon; epsilon at adaptive stop is diagnostic only',
    secure_mode=False, research_outputs_are_dp_releases=False,
    multiple_models='Per-model bounds; joint release requires composition; control is non-private',
    n_shadow_models=N_SHADOW_MODELS, shadow_algorithms=SHADOW_ALGORITHMS, shadow_seeds=shadow_seeds,
    shadow_models_per_algorithm=4, shadow_mlp_config=SHADOW_MLP_CONFIG,
    records_per_class_per_shadow=SHADOW_RECORDS_PER_CLASS,
    attack='Frozen complete-feature RF transfer attack; non-DP tree and generic MLP shadows',
    attack_features=list(attack_result['classifier'].feature_names_in_), include_blind_walk=True,
    walk_config=WALK_CONFIG, extra_numeric_protocol=EXPANDED_NUMERIC_WALK,
    walk_scope='Consistent extra numeric protocol for shadows and targets',
    shadow_query_audits=shadow_query_audits, auxiliary_attack_metrics=attack_result['metrics'],
    accuracy_threshold=attack_result['accuracy_threshold'], low_fpr_threshold=attack_result['low_fpr_threshold'],
    plots='Equally spaced categorical requested epsilon labels; non-private horizontal reference',
    versions=versions, checkpoints_enabled=False, logging_enabled=False,
    missing_feature_experiments=False, repeated_seed_confirmation=False)
_ = (RUN_DIR / 'protocol.json').write_text(json.dumps(json_value(protocol), indent=2, allow_nan=False), encoding='utf-8')
print('Final metrics:', RUN_DIR / 'dp_target_and_mia_metrics.csv')
print('Graphs: target_accuracy_vs_epsilon, mia_accuracy_vs_epsilon, mia_low_fpr_vs_epsilon (PNG and SVG)')


Implementation references:
[Opacus PrivacyEngine](https://opacus.ai/api/privacy_engine.html),
[BatchMemoryManager](https://opacus.ai/api/batch_memory_manager.html),
[DPDataLoader](https://opacus.ai/api/data_loader.html),
[Opacus FAQ](https://opacus.ai/docs/faq).
